# Faz 7b — 5-Fold Cross-Validation (SwinTransformer)

Her fold için dört konfigürasyonu eğitir ve test eder:
1. Baseline (BCE)
2. FocalBCE
3. FocalBCE + WeightedSampler
4. FocalBCE + WeightedSampler + ThreshTuning

**Çıktı:** `outputs/results/crossval_results.json` → Faz8'de Fig 11 için kullanılır.

In [ ]:
import os, sys, json, time
from pathlib import Path

IS_COLAB  = 'google.colab' in sys.modules
IS_KAGGLE = os.environ.get('KAGGLE_KERNEL_RUN_TYPE') is not None
IS_LOCAL  = not IS_COLAB and not IS_KAGGLE

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/abdomen')
    DATA_DIR     = Path('/content/drive/MyDrive/abdomenDataSet')
    WORK_DIR     = Path('/content')
    DRIVE_BASE   = None
elif IS_KAGGLE:
    PROJECT_ROOT = Path('/kaggle/working/abdomen')
    DATA_DIR     = Path('/kaggle/input/abdomen-dataset')
    WORK_DIR     = Path('/kaggle/working')
    DRIVE_BASE   = None
elif IS_LOCAL:
    try:
        from dotenv import load_dotenv; load_dotenv()
    except ImportError:
        pass
    _proje       = Path(os.environ.get('TR_ABDOMEN_PROJE',  r'D:/makale-pdf/Proje'))
    DATA_DIR     = Path(os.environ.get('TR_ABDOMEN_BASE',   str(_proje / 'abdomen')))
    WORK_DIR     = Path(os.environ.get('ABDOMEN_OUT_DIR',   str(_proje / 'outputs')))
    PROJECT_ROOT = DATA_DIR
    DRIVE_BASE   = None

SPLIT_DIR   = Path(os.environ.get('ABDOMEN_SPLIT_DIR', str(WORK_DIR / 'splits')))      if IS_LOCAL else WORK_DIR / 'splits'
CKPT_DIR    = Path(os.environ.get('ABDOMEN_CKPT_DIR',  str(WORK_DIR / 'checkpoints'))) if IS_LOCAL else WORK_DIR / 'checkpoints'
CKPT_DIR    = CKPT_DIR / 'crossval'
RESULTS_DIR = WORK_DIR / 'results'
CKPT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT))
os.environ.setdefault('ABDOMEN_PROJECT_ROOT', str(PROJECT_ROOT))
os.environ.setdefault('ABDOMEN_SPLIT_DIR',    str(SPLIT_DIR))

N_FOLDS = 5
SUPER_CLASSES = [
    'acute_cholecystitis', 'kidney_ureter_stone', 'acute_pancreatitis',
    'aortic_aneurysm_dissection', 'acute_appendicitis', 'acute_diverticulitis',
]

print(f'Ortam: {"LOCAL" if IS_LOCAL else "COLAB" if IS_COLAB else "KAGGLE"}')
print(f'DATA_DIR   : {DATA_DIR}')
print(f'WORK_DIR   : {WORK_DIR}')
print(f'SPLIT_DIR  : {SPLIT_DIR}')
print(f'CKPT_DIR   : {CKPT_DIR}')

In [ ]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {DEVICE}')

## Konfigürasyonlar

In [ ]:
from src.config import ClsConfig

# 4 konfigürasyon — her biri ayrı anahtar ile
CV_CONFIGS = [
    {
        'name':               'Baseline (BCE)',
        'use_focal':          False,
        'use_weighted_sampler': False,
        'tune_thresholds':    False,
    },
    {
        'name':               'FocalBCE',
        'use_focal':          True,
        'use_weighted_sampler': False,
        'tune_thresholds':    False,
    },
    {
        'name':               'FocalBCE+Sampler',
        'use_focal':          True,
        'use_weighted_sampler': True,
        'tune_thresholds':    False,
    },
    {
        'name':               'FocalBCE+Sampler+Thresh',
        'use_focal':          True,
        'use_weighted_sampler': True,
        'tune_thresholds':    True,
    },
]

BASE_CFG = dict(
    backbone      = 'swin_base_patch4_window12_384.ms_in22k_ft_in1k',
    num_classes   = 6,
    input_size    = 384,
    batch_size    = 16,
    epochs        = 50,
    lr            = 2e-4,
    weight_decay  = 1e-4,
    warmup_epochs = 3,
    focal_gamma   = 2.0,
    accum_steps   = 2,
)

print(f'{len(CV_CONFIGS)} konfigürasyon × {N_FOLDS} fold = {len(CV_CONFIGS)*N_FOLDS} eğitim')

## Eğitim + Değerlendirme Döngüsü

In [ ]:
from src.train_cls import build_model, train_one_fold, tune_thresholds
from src.splits    import load_fold

def eval_fold(model, df_val, thresholds):
    """Validation setinde Top-5 Mean F1 hesaplar."""
    import torchvision.transforms as T
    from src.datasets import AbdomenClsDataset

    transform = T.Compose([
        T.Resize((384, 384)), T.ToTensor(),
        T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
    ])
    loader = DataLoader(
        AbdomenClsDataset(df_val, transform=transform),
        batch_size=32, shuffle=False, num_workers=2
    )
    all_preds, all_labels = [], []
    model.eval()
    with torch.no_grad():
        for batch in loader:
            imgs   = batch['image'].to(DEVICE)
            probs  = torch.sigmoid(model(imgs)).cpu().numpy()
            all_preds.append(probs)
            all_labels.append(batch['labels'].numpy())
    preds  = np.concatenate(all_preds)
    labels = np.concatenate(all_labels)

    per_class_f1 = {}
    for i, cls in enumerate(SUPER_CLASSES):
        tau     = thresholds.get(cls, 0.5)
        pred_b  = (preds[:, i] >= tau).astype(int)
        tp = int(((pred_b==1)&(labels[:,i]==1)).sum())
        fp = int(((pred_b==1)&(labels[:,i]==0)).sum())
        fn = int(((pred_b==0)&(labels[:,i]==1)).sum())
        prec = tp/max(tp+fp,1); rec = tp/max(tp+fn,1)
        per_class_f1[cls] = round(2*prec*rec/max(prec+rec,1e-9), 4)
    return round(float(np.mean(list(per_class_f1.values()))), 4), per_class_f1


# Sonuçları tutacak yapı
cv_results = {
    'configs': [c['name'] for c in CV_CONFIGS],
    'folds':   list(range(N_FOLDS)),
    'per_fold_f1':    {c['name']: [] for c in CV_CONFIGS},
    'per_fold_per_class': {c['name']: [] for c in CV_CONFIGS},
}

total = len(CV_CONFIGS) * N_FOLDS
done  = 0

for cfg in CV_CONFIGS:
    cfg_name = cfg['name']
    print(f'\n══ Konfigürasyon: {cfg_name} ══')

    for fold in range(N_FOLDS):
        done += 1
        print(f'  [{done}/{total}] Fold {fold} başlıyor...')

        ckpt_path = CKPT_DIR / f'{cfg_name.replace(" ","_")}_fold{fold}_best.pth'

        # Eğer checkpoint zaten varsa atla
        if ckpt_path.exists():
            print(f'    Checkpoint mevcut, yükleniyor: {ckpt_path.name}')
            train_cfg = ClsConfig(
                **{**BASE_CFG,
                   'use_focal': cfg['use_focal'],
                   'use_weighted_sampler': cfg['use_weighted_sampler'],
                   'fold': fold}
            )
            model = build_model(train_cfg).to(DEVICE)
            ckpt  = torch.load(ckpt_path, map_location=DEVICE)
            model.load_state_dict(ckpt.get('model_state_dict', ckpt))
        else:
            # Eğit
            train_cfg = ClsConfig(
                **{**BASE_CFG,
                   'use_focal': cfg['use_focal'],
                   'use_weighted_sampler': cfg['use_weighted_sampler'],
                   'fold': fold,
                   'ckpt_dir': str(CKPT_DIR)}
            )
            model = train_one_fold(train_cfg)
            # En iyi checkpoint'i kopyala
            trained_ckpt = CKPT_DIR / f'fold{fold}_best.pth'
            if trained_ckpt.exists():
                trained_ckpt.rename(ckpt_path)

        # Threshold tuning
        df_val = load_fold(fold, 'val')
        if cfg['tune_thresholds']:
            thresholds = tune_thresholds(model, df_val, SUPER_CLASSES, DEVICE)
            thresh_path = CKPT_DIR / f'{cfg_name.replace(" ","_")}_fold{fold}_thresh.json'
            with open(thresh_path, 'w') as fh:
                json.dump(thresholds, fh, indent=2)
        else:
            thresholds = {c: 0.5 for c in SUPER_CLASSES}

        # Değerlendirme (val seti)
        f1, per_cls = eval_fold(model, df_val, thresholds)
        cv_results['per_fold_f1'][cfg_name].append(f1)
        cv_results['per_fold_per_class'][cfg_name].append(per_cls)
        print(f'    Fold {fold} Top-5 F1 = {f1:.4f}')

        # Belleği temizle
        del model
        if DEVICE == 'cuda': torch.cuda.empty_cache()

# Özet
print('\n══ 5-Fold CV Özeti ══')
for cfg in CV_CONFIGS:
    name   = cfg['name']
    scores = cv_results['per_fold_f1'][name]
    print(f'  {name}: {np.mean(scores):.4f} ± {np.std(scores):.4f}  (folds: {scores})')

# JSON'a kaydet
out_path = RESULTS_DIR / 'crossval_results.json'
with open(out_path, 'w') as f:
    json.dump(cv_results, f, indent=2)
print(f'\nKaydedildi: {out_path}')
print('Faz8_Makale_Analiz.ipynb çalıştırarak Fig 11 oluşturun.')

## Hızlı Önizleme (isteğe bağlı)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'font.size': 11, 'figure.dpi': 120})

configs = cv_results['configs']
palette = ['#9E9E9E', '#4C72B0', '#55A868', '#C44E52'][:len(configs)]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle('5-Fold Cross-Validation — Quick Preview', fontsize=13, fontweight='bold')

means = [np.mean(cv_results['per_fold_f1'][c]) for c in configs]
stds  = [np.std(cv_results['per_fold_f1'][c])  for c in configs]
x = range(len(configs))
axes[0].bar(x, means, yerr=stds, color=palette, edgecolor='black',
            linewidth=0.5, alpha=0.88, capsize=5)
axes[0].set_xticks(x)
axes[0].set_xticklabels(configs, rotation=20, ha='right', fontsize=9)
axes[0].set_ylabel('Top-5 Mean F1')
axes[0].set_ylim(0, 1)
axes[0].set_title('Mean ± Std')

for i, cfg in enumerate(configs):
    axes[1].plot(cv_results['folds'], cv_results['per_fold_f1'][cfg],
                 'o-', color=palette[i], label=cfg, linewidth=1.8)
axes[1].set_xlabel('Fold'); axes[1].set_ylabel('Top-5 Mean F1')
axes[1].set_ylim(0, 1); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
axes[1].set_title('Per-Fold')

plt.tight_layout()
plt.show()